<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-03-prompt-engineering/lesson-3.3-advanced-reasoning/notebooks/exercises-3_3.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 3.3: Advanced Reasoning: CoT, ToT, ReAct, Reflexion

*Module 3 · Prompt Engineering & Context Design*

> The same model can score 30% or 90% on the same problem. The difference? How you ask it to think . This lesson teaches 10 reasoning techniques — from "think step by step" to reasoning models that learned to think via reinforcement learning.

---

## About this notebook

This notebook contains the **12 runnable code examples** from the published lesson `lesson-3.3.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. `setup.py`
2. Step 1 — The CoT Family — 9 Techniques, Not Just One — `cot_taxonomy.py`
3. Step 2 — Chain-of-Thought — "Show Your Work" — `cot_proof.py`
4. Step 2 — Chain-of-Thought — "Show Your Work" — `cot_class.py`
5. Step 3 — Reasoning Models — o3, DeepSeek-R1, Claude Extended Thinking — `reasoning_models.py`
6. Step 4 — When NOT to Use Reasoning — The Most Counterintuitive Lesson — `when_not_to_reason.py`
7. Step 5 — Structured CoT & Prompt Decomposition — Production Patterns — `structured_cot.py`
8. Step 6 — Tree-of-Thought — "Explore 3 Paths, Pick the Best" — `tot_class.py`
9. Step 7 — ReAct — "Think, Then Use Tools, Then Think Again" — `react_class.py`
10. Step 8 — Reflexion — "Try, Critique, Improve, Repeat" — `reflexion_class.py`
11. Step 9 — Cost & Latency Decision Framework — `cost_framework.py`
12. Step 10 — Verification Chains — Generate, Check, Correct — `verification.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


**`setup.py`** · _Block 1 of 12_


In [ ]:
import google.generativeai as genai
import os, json, re

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

def ask(prompt, temp=0.3, system=None):
    model = genai.GenerativeModel("gemini-2.0-flash",
        system_instruction=system,
        generation_config={"temperature": temp})
    return model.generate_content(prompt).text.strip()


### Step 1 · The CoT Family — 9 Techniques, Not Just One

Chain-of-thought is not one technique. It is a family of 9 methods across 3 generations of research.

**`cot_taxonomy.py`** · _Block 2 of 12_


In [ ]:
print("""
THE COT FAMILY (9 TECHNIQUES):

GENERATION 1 — Prompting-based (2022):
  1. Few-shot CoT (Wei et al. NeurIPS 2022)
     → Give examples WITH reasoning chains
     → PaLM-540B: state-of-the-art on GSM8K math
     → Only works reliably in models >100B parameters

  2. Zero-shot CoT (Kojima et al. NeurIPS 2022)
     → Just add "Let's think step by step"
     → MultiArith: 17.7% → 78.7% accuracy!
     → No examples needed. Task-agnostic.

  3. Self-Consistency (Wang et al. ICLR 2023)
     → Sample N chains at temperature>0, majority vote
     → GSM8K: +17.9% over vanilla CoT
     → Trade-off: N× cost for much higher reliability

GENERATION 2 — Structured reasoning (2022-2023):
  4. Plan-and-Solve — "Devise a plan, then execute"
  5. Least-to-Most — Decompose into simpler subproblems
  6. Tree-of-Thought — Explore multiple paths, prune bad ones
  7. Meta-Prompting — LLM chooses its own reasoning strategy

GENERATION 3 — Native reasoning (2024-2026):
  8. Extended Thinking — Claude budget_tokens, o3 reasoning_effort
  9. Reinforcement-learned reasoning — DeepSeek-R1, QwQ
     → Model learns to reason via RL, NOT prompting
     → "Think step by step" is REDUNDANT and harmful!
""")


> **What just happened?** CoT is a family of 9 techniques across 3 generations. Generation 1: prompt-based (zero-shot, few-shot, self-consistency). Generation 2: structured (plan-and-solve, least-to-most, ToT, meta-prompting). Generation 3: native reasoning models that learned to reason via RL. The biggest shift: Generation 3 models make explicit CoT prompting partially obsolete.


### Step 2 · Chain-of-Thought — "Show Your Work"

The simplest and most powerful reasoning upgrade. Just add "think step by step."

**`cot_proof.py`** · _Block 3 of 12_

PROOF: Chain-of-Thought fixes wrong answers


In [ ]:
# =============================================
# PROOF: Chain-of-Thought fixes wrong answers
# =============================================

problem = """A store sells apples at ₹40 each. On weekends, they offer 
buy-2-get-1-free. Rahul buys 7 apples on a Saturday. How much does he pay?"""

# ❌ WITHOUT CoT — AI often jumps to wrong answer
direct = f"""{problem}
Give only the final answer in rupees."""

# ✅ WITH CoT — AI shows every step
cot = f"""{problem}
Solve this step by step:
Step 1: Understand the offer
Step 2: Figure out how many free apples Rahul gets
Step 3: Calculate how many he actually pays for
Step 4: Multiply to get total cost
Show all steps, then give the final answer."""

print("═══ WITHOUT Chain-of-Thought ═══")
print(ask(direct))
# Often says ₹280 (7 × 40) — WRONG! Ignores the offer.
# Or says ₹200 — also wrong.

print("\n═══ WITH Chain-of-Thought ═══")
print(ask(cot))
# Step 1: Buy 2, get 1 free means every 3 apples, pay for 2
# Step 2: 7 apples = 2 groups of 3 + 1 extra. Free apples: 2
# Step 3: Pay for 7 - 2 = 5 apples
# Step 4: 5 × ₹40 = ₹200 ✅


> **What just happened?** Without CoT, the AI often gives the wrong answer because it tries to compute everything in one leap. With CoT, it breaks the problem into 4 visible steps, catches the "buy-2-get-1-free" logic, and arrives at the correct ₹200. Adding "think step by step" can improve accuracy on reasoning tasks by 20-40%.


**`cot_class.py`** · _Block 4 of 12_

CHAIN-OF-THOUGHT as a reusable class


In [ ]:
# =============================================
# CHAIN-OF-THOUGHT as a reusable class
# =============================================

class ChainOfThought:
    """Force the AI to reason step-by-step before answering."""
    
    def __init__(self, system_prompt=None):
        self.system = system_prompt or "You are a careful problem solver. Always think step by step."
    
    def solve(self, problem: str, num_steps: int = 4) -> dict:
        """Solve a problem with explicit step-by-step reasoning."""
        
        prompt = f"""Problem: {problem}

Solve step by step. Use exactly this format:

Step 1: [first step of reasoning]
Step 2: [second step]
Step 3: [third step]
{"Step 4: [fourth step]" if num_steps >= 4 else ""}

FINAL ANSWER: [your answer]"""
        
        response = ask(prompt, system=self.system)
        
        # Parse steps and final answer
        steps = re.findall(r"Step \d+:(.+?)(?=Step \d+:|FINAL ANSWER:|$)", response, re.DOTALL)
        answer_match = re.search(r"FINAL ANSWER:(.+)", response, re.DOTALL)
        
        return {
            "steps": [s.strip() for s in steps],
            "answer": answer_match.group(1).strip() if answer_match else response,
            "full_response": response,
        }

# ── Use it ──
cot = ChainOfThought()

problems = [
    "If a train travels 60 km/h for 2.5 hours, then 80 km/h for 1.5 hours, what is the total distance?",
    "A shirt costs ₹800. There's a 25% discount, then an additional 10% off on the discounted price. What's the final price?",
    "There are 3 boxes. Box A has 2 red and 3 blue balls. Box B has 4 red and 1 blue. Box C has 1 red and 5 blue. Which box has the highest proportion of red balls?",
]

for i, prob in enumerate(problems):
    print(f"\nProblem {i+1}: {prob[:60]}...")
    result = cot.solve(prob)
    
    for j, step in enumerate(result["steps"]):
        print(f"  Step {j+1}: {step[:80]}")
    print(f"  ANSWER: {result['answer']}")
    print("  ─────────────")


### Step 3 · Reasoning Models — o3, DeepSeek-R1, Claude Extended Thinking

2025 changed everything. Models that learned to reason via RL, not prompting.

**`reasoning_models.py`** · _Block 5 of 12_


In [ ]:
print("""
REASONING MODELS (2025-2026):

  OpenAI o3 / o4-mini (April 2025):
    → 93.4% on AIME 2024 (top-percentile human math)
    → Native tool use WITHIN reasoning chain
    → reasoning_effort: low / medium / high
    → DO NOT say "think step by step" — it HURTS performance

  DeepSeek-R1 (January 2025):
    → Open-source MIT license, 671B MoE (37B active)
    → Visible reasoning in <think> tags
    → $0.55/M tokens vs o1 at $15 — 27x cheaper!
    → Distilled versions: 1.5B to 70B on Hugging Face

  Claude Extended Thinking (February 2025):
    → budget_tokens parameter (min 1024) controls depth
    → Performance improves logarithmically with budget
    → Claude 4.6+: adaptive thinking (auto-calibrates)
    → Interleaved thinking between tool calls

  Gemini 2.5 Flash (June 2025):
    → thinkingBudget: 0 to 24,576 tokens (or -1 for auto)
    → Budget=0 STILL outperforms Gemini 2.0 Flash
    → $0.30/M input — best price/performance ratio

  QwQ-32B (March 2025):
    → Matches DeepSeek-R1 671B with only 32B params!
    → Self-hostable on a single GPU
    → Apache 2.0 license
""")


> **What just happened?** Five reasoning models changed prompt engineering in 2025: o3/o4-mini (strongest, expensive), DeepSeek-R1 (open-source, 27x cheaper), Claude extended thinking (budget control), Gemini 2.5 Flash (best price/performance), QwQ-32B (self-hostable). All learned reasoning via RL — explicit CoT prompting is redundant and sometimes harmful.


### Step 4 · When NOT to Use Reasoning — The Most Counterintuitive Lesson

Wharton 2025: CoT adds only 2.9-3.1% accuracy on reasoning models but 20-80% latency.

**`when_not_to_reason.py`** · _Block 6 of 12_


In [ ]:
print("""
WHEN TO SKIP REASONING (use standard model, no CoT):
  ✗ Simple classification, sentiment analysis
  ✗ Translation, summarization
  ✗ Factual lookup / retrieval
  ✗ Content generation (creative writing)
  ✗ Basic arithmetic (addition, multiplication)

WHEN TO USE CoT PROMPTING (standard model + CoT):
  ✓ Multi-step math with word problems
  ✓ Logic puzzles, constraint satisfaction
  ✓ Complex analysis (compare 3+ options)
  ✓ Indian tax calculations (GST slab + CGST/SGST split)

WHEN TO USE REASONING MODELS (o3, R1, Claude thinking):
  ✓ Competition-level math (AIME, Olympiad)
  ✓ Complex code generation & debugging
  ✓ Legal reasoning with Indian statutes
  ✓ Multi-step planning & strategy

CRITICAL ANTI-PATTERNS:
  ✗ "Think step by step" on o3/R1 → DEGRADES performance
  ✗ Few-shot examples on reasoning models → format imitation
  ✗ CoT on Gemini 2.5 Flash → WORSE results (-3.3%)
  ✗ Reasoning for simple tasks → 20-80% more latency for <3% gain

  The Wharton study (June 2025): CoT on reasoning models
  gives only +2.9-3.1% accuracy but adds 20-80% latency.
""")


> **What just happened?** The most counterintuitive lesson: adding CoT to reasoning models hurts performance . Wharton 2025 showed only +2.9-3.1% accuracy for 20-80% more latency. On Gemini 2.5 Flash, CoT made results worse by 3.3% . Match reasoning depth to task complexity: standard model for simple tasks, CoT for medium, reasoning models for hard.


### Step 5 · Structured CoT & Prompt Decomposition — Production Patterns

Separate reasoning from answers with tags. Chain multiple calls for complex tasks.

**`structured_cot.py`** · _Block 7 of 12_

STRUCTURED CoT — XML tags for production


In [ ]:
# =============================================
# STRUCTURED CoT — XML tags for production
# =============================================

structured = """Calculate the total GST for this invoice.

Items: 3 laptops at ₹45,000 each (18% GST, intra-state)

Work through this inside <thinking> tags.
Give ONLY the final breakdown inside <answer> tags.

<thinking>
[your step-by-step reasoning]
</thinking>
<answer>
[Base: ₹X | CGST 9%: ₹Y | SGST 9%: ₹Z | Total: ₹W]
</answer>"""

# PROMPT DECOMPOSITION — Chain multiple calls
# Step 1: Extract invoice items
# Step 2: Determine GST slab per item
# Step 3: Calculate CGST/SGST/IGST split
# Step 4: Compute totals with ITC
# Each step validates before passing to next

print("""
STRUCTURED CoT:
  Put <thinking> BEFORE <answer> — forces reasoning first
  Parse <answer> tags programmatically for downstream use
  Log <thinking> for debugging without showing to users

PROMPT DECOMPOSITION:
  4 focused calls > 1 complex call (ACL 2024)
  Each step has validation before passing output
  Easier to debug, test, and improve individual steps
  This is how production AI systems actually work.
""")


> **What just happened?** Structured CoT uses XML tags (<thinking> + <answer>) to separate reasoning from output — essential for production parsing. Prompt decomposition chains multiple focused API calls instead of one complex prompt. ACL 2024 confirmed: 4 sequential focused prompts outperform 1 complex prompt. Both patterns are how production AI systems actually work.


### Step 6 · Tree-of-Thought — "Explore 3 Paths, Pick the Best"

When there's no single right approach, try multiple and evaluate.

**`tot_class.py`** · _Block 8 of 12_

TREE-OF-THOUGHT: Explore multiple approaches — 3 API calls to generate ideas, 1 to judge.


In [ ]:
# =============================================
# TREE-OF-THOUGHT: Explore multiple approaches
# 3 API calls to generate ideas, 1 to judge.
# =============================================

class TreeOfThought:
    """Generate multiple approaches, evaluate each, pick the best."""
    
    def __init__(self, num_branches: int = 3):
        self.num_branches = num_branches
    
    def generate_approaches(self, problem: str) -> list[str]:
        """Step 1: Ask the AI for multiple different approaches."""
        prompt = f"""Problem: {problem}

Generate exactly {self.num_branches} DIFFERENT approaches to solve this.
For each approach, think it through step-by-step to a conclusion.

Approach 1:
[Think through this approach step by step to a final answer]

Approach 2:
[A DIFFERENT method — think through to a final answer]

Approach 3:
[Yet ANOTHER different method — think through to a final answer]"""
        
        response = ask(prompt, temp=0.7)  # higher temp = more diverse ideas
        
        # Split into approaches
        approaches = re.split(r"Approach \d+:", response)
        approaches = [a.strip() for a in approaches if a.strip()]
        return approaches[:self.num_branches]
    
    def evaluate_and_pick(self, problem: str, approaches: list[str]) -> dict:
        """Step 2: Judge which approach is best."""
        approaches_text = ""
        for i, a in enumerate(approaches):
            approaches_text += f"\nApproach {i+1}:\n{a}\n"
        
        prompt = f"""Problem: {problem}

Three approaches were proposed:
{approaches_text}

Evaluate each approach:
1. Is the reasoning correct?
2. Is the final answer right?
3. Which approach is most reliable?

Pick the BEST approach and give the final answer.

Best approach number: [1, 2, or 3]
Reason: [why this one is best]
Final answer: [the answer]"""
        
        response = ask(prompt, temp=0.1)  # low temp for judging
        return {"evaluation": response, "approaches": approaches}
    
    def solve(self, problem: str) -> dict:
        """Full pipeline: generate → evaluate → answer."""
        print(f"Generating {self.num_branches} approaches...")
        approaches = self.generate_approaches(problem)
        
        for i, a in enumerate(approaches):
            print(f"\n  Approach {i+1}: {a[:100]}...")
        
        print("\nEvaluating approaches...")
        result = self.evaluate_and_pick(problem, approaches)
        print(f"\n{result['evaluation']}")
        return result

# ── Test on a planning problem ──
tot = TreeOfThought(num_branches=3)

planning_problem = """A startup has ₹5 lakhs budget and 3 months to build an MVP 
for a food delivery app in Hyderabad. They have 2 developers and 
1 designer. What should their development strategy be?"""

result = tot.solve(planning_problem)

# The AI generates 3 different strategies (e.g., "build from scratch",
# "use a template", "outsource backend"), evaluates each for cost,
# time, and quality, then picks the best one with reasoning.


> **What just happened?** We made 2 API calls: one to generate 3 different approaches (with higher temperature for diversity), and one to judge which approach is best (with lower temperature for careful evaluation). For the startup planning problem, the AI might generate: (1) build from scratch with React Native, (2) use a no-code platform, (3) fork an open-source template. Then it evaluates cost, timeline, and quality of each, and recommends the best. This is dramatically better than a single answer for complex, open-ended problems.


### Step 7 · ReAct — "Think, Then Use Tools, Then Think Again"

The AI can't know today's weather or stock prices. ReAct lets it use tools to find out.

**`react_class.py`** · _Block 9 of 12_

ReAct: AI that can THINK and USE TOOLS — We give the AI access to tools (calculator,


In [ ]:
# =============================================
# ReAct: AI that can THINK and USE TOOLS
# We give the AI access to tools (calculator,
# search, etc.) and let it decide when to use them.
# =============================================

import math

class ReActAgent:
    """An AI agent that reasons and acts using tools."""
    
    def __init__(self):
        # Define available tools
        self.tools = {
            "calculator": self._calculator,
            "unit_convert": self._unit_convert,
            "lookup": self._lookup,
        }
        self.tool_descriptions = """Available tools:
- calculator(expression): Evaluate a math expression. Example: calculator(25 * 4 + 10)
- unit_convert(value, from_unit, to_unit): Convert units. Example: unit_convert(100, km, miles)
- lookup(topic): Look up a fact. Example: lookup(population of Hyderabad)"""
    
    def _calculator(self, expression):
        try:
            # Safe math evaluation
            allowed = {"math": math, "abs": abs, "round": round}
            result = eval(expression, {"__builtins__": {}}, allowed)
            return str(result)
        except:
            return "Error: invalid expression"
    
    def _unit_convert(self, value, from_unit, to_unit):
        conversions = {
            ("km", "miles"): 0.621371,
            ("kg", "lbs"): 2.20462,
            ("celsius", "fahrenheit"): lambda v: v * 9/5 + 32,
            ("inr", "usd"): 0.012,
        }
        key = (from_unit.lower(), to_unit.lower())
        if key in conversions:
            factor = conversions[key]
            result = factor(float(value)) if callable(factor) else float(value) * factor
            return f"{value} {from_unit} = {result:.2f} {to_unit}"
        return f"Cannot convert {from_unit} to {to_unit}"
    
    def _lookup(self, topic):
        # In production, this would call a search API
        facts = {
            "population of hyderabad": "approximately 10.5 million (2024)",
            "gst rate india": "standard GST rates: 5%, 12%, 18%, 28%",
            "minimum wage india": "varies by state, ₹178-₹700/day",
        }
        return facts.get(topic.lower(), f"No data found for '{topic}'")
    
    def _execute_tool(self, action_text):
        """Parse and execute a tool call from the AI's response."""
        match = re.match(r"(\w+)\((.+)\)", action_text.strip())
        if not match:
            return "Could not parse tool call"
        
        tool_name = match.group(1)
        args = [a.strip().strip("'\"") for a in match.group(2).split(",")]
        
        if tool_name in self.tools:
            return self.tools[tool_name](*args)
        return f"Unknown tool: {tool_name}"
    
    def solve(self, question: str, max_steps: int = 5) -> str:
        """Run the Thought → Action → Observation loop."""
        
        history = f"""Answer this question using the available tools.

{self.tool_descriptions}

Use this EXACT format for each step:
Thought: [what I need to figure out next]
Action: [tool_name(arguments)]

When you have enough info, use:
Thought: I now have enough information.
Final Answer: [your answer]

Question: {question}
"""
        
        print(f"Question: {question}\n")
        
        for step in range(max_steps):
            # Ask the AI to think and choose an action
            response = ask(history, temp=0.1)
            
            # Check if it has a final answer
            if "Final Answer:" in response:
                answer = response.split("Final Answer:")[-1].strip()
                print(f"  Step {step+1}: FINAL ANSWER")
                print(f"  → {answer}")
                return answer
            
            # Extract thought and action
            thought_match = re.search(r"Thought:(.+?)(?=Action:|$)", response, re.DOTALL)
            action_match = re.search(r"Action:(.+?)(?=\n|$)", response)
            
            thought = thought_match.group(1).strip() if thought_match else "..."
            action = action_match.group(1).strip() if action_match else ""
            
            print(f"  Step {step+1}:")
            print(f"    Thought: {thought[:80]}")
            print(f"    Action:  {action}")
            
            # Execute the tool and get observation
            observation = self._execute_tool(action)
            print(f"    Result:  {observation}")
            
            # Add everything to history for next round
            history += f"\nThought: {thought}\nAction: {action}\nObservation: {observation}\n"
        
        return "Could not solve within max steps"

# ── Test it! ──
agent = ReActAgent()

agent.solve("If I have ₹50,000 and the GST rate is 18%, what is the total cost including GST?")
# Step 1: Thought: I need to calculate GST on ₹50,000
#         Action: calculator(50000 * 0.18)
#         Result: 9000.0
# Step 2: Thought: GST is ₹9,000, total = 50000 + 9000
#         Action: calculator(50000 + 9000)
#         Result: 59000
# Step 3: FINAL ANSWER: ₹59,000 (₹50,000 + ₹9,000 GST)


> **What just happened?** The AI ran a Thought → Action → Observation loop: it thought about what it needed, called a tool (calculator), saw the result, thought again, called another tool, and finally gave the answer. It made 2 tool calls and 3 thinking steps. This is the foundation of AI agents — the same pattern used by LangChain, AutoGPT, and every tool-using AI system. In production, the "tools" would be web search, databases, APIs, and more.


### Step 8 · Reflexion — "Try, Critique, Improve, Repeat"

The AI writes a first draft, critiques its own work, then rewrites. Each iteration improves.

**`reflexion_class.py`** · _Block 10 of 12_

REFLEXION: Generate → Critique → Improve — Each loop makes the output better.


In [ ]:
# =============================================
# REFLEXION: Generate → Critique → Improve
# Each loop makes the output better.
# =============================================

class Reflexion:
    """Iteratively improve output through self-critique."""
    
    def __init__(self, max_iterations: int = 3):
        self.max_iterations = max_iterations
    
    def generate(self, task: str) -> str:
        """Step 1: Create a first attempt."""
        return ask(f"Complete this task:\n{task}", temp=0.5)
    
    def critique(self, task: str, attempt: str) -> dict:
        """Step 2: Critique the attempt honestly."""
        prompt = f"""Task: {task}

Current attempt:
\"\"\"
{attempt}
\"\"\"

Critically evaluate this attempt. Be harsh but constructive.
Score it 1-10 and explain specific weaknesses.

Use this exact format:
Score: [1-10]
Strengths: [what's good]
Weaknesses: [what's bad — be specific]
Suggestions: [concrete improvements]"""
        
        response = ask(prompt, temp=0.2)
        
        # Extract score
        score_match = re.search(r"Score:\s*(\d+)", response)
        score = int(score_match.group(1)) if score_match else 5
        
        return {"score": score, "feedback": response}
    
    def improve(self, task: str, attempt: str, feedback: str) -> str:
        """Step 3: Rewrite based on critique."""
        prompt = f"""Task: {task}

Previous attempt:
\"\"\"
{attempt}
\"\"\"

Feedback on the previous attempt:
{feedback}

Now write an IMPROVED version that addresses ALL the feedback.
Keep what was good, fix what was bad."""
        
        return ask(prompt, temp=0.4)
    
    def solve(self, task: str, target_score: int = 8) -> dict:
        """Full loop: generate → critique → improve → repeat."""
        
        print(f"Task: {task[:80]}...\n")
        
        # First attempt
        current = self.generate(task)
        history = []
        
        for i in range(self.max_iterations):
            # Critique
            review = self.critique(task, current)
            score = review["score"]
            history.append({"iteration": i+1, "score": score, "output": current[:100]})
            
            print(f"  Iteration {i+1}: Score {score}/10")
            
            # Good enough? Stop.
            if score >= target_score:
                print(f"  ✅ Reached target score ({target_score})! Stopping.")
                break
            
            # Not good enough? Improve.
            print(f"  Improving based on feedback...")
            current = self.improve(task, current, review["feedback"])
        
        return {"final_output": current, "history": history, "iterations": len(history)}

# ── Test: Write a professional email ──
reflexion = Reflexion(max_iterations=3)

result = reflexion.solve(
    task="""Write a professional email from a startup founder to a potential 
investor requesting a meeting. The startup is an AI-powered crop 
disease detection app for Indian farmers. Include: the problem, 
the solution, traction (10,000 farmers in 3 states), and the ask 
(₹2 crore seed funding). Keep it under 200 words.""",
    target_score=8
)

print(f"\n═══ FINAL OUTPUT (after {result['iterations']} iterations) ═══")
print(result["final_output"])

print("\nScore progression:")
for h in result["history"]:
    print(f"  Iteration {h['iteration']}: {h['score']}/10")
# Typical: 5/10 → 7/10 → 9/10. Each iteration visibly improves!


> **What just happened?** The AI wrote a first-draft email (scored ~5/10), then critiqued itself ("the opening is generic, the ask is vague, the tone is too casual"), then rewrote fixing those issues (scored ~7/10), critiqued again ("better but could include a specific ROI number"), rewrote again (scored ~9/10), and stopped. Three API calls for generate + critique + improve, repeated until the score hits 8+. The final email is dramatically better than the first attempt. This pattern works for any creative or writing task.


### Step 9 · Cost & Latency Decision Framework

DeepSeek-R1 is 27x cheaper than o1. Match model to task.

**`cost_framework.py`** · _Block 11 of 12_


In [ ]:
print("""
COST COMPARISON (per million tokens, 2025-2026):
  Simple tasks:    GPT-4.1-mini $0.10 / Gemini Flash-Lite $0.02
  Medium reasoning: o4-mini $1.10 / Gemini 2.5 Flash $0.30
  Hard reasoning:  o3 $10-15 / Claude thinking $3-15
  Open-source:     DeepSeek-R1 $0.55 / QwQ-32B free (self-host)

LATENCY REALITIES:
  Standard model:       0.5-2s response
  Standard + CoT:       1-4s (2-3x from extra tokens)
  Reasoning model:      3-11s time-to-first-token BEFORE output
  Reasoning + CoT:      +10-20s additional (Wharton study)

DECISION FRAMEWORK:
  1. Can a standard model do it? → Use GPT-4.1-mini/Flash
  2. Needs multi-step reasoning? → Add CoT to standard model
  3. Still failing? → Use o4-mini medium or Gemini Flash thinking
  4. Competition-level hard? → Use o3 high or Claude extended
  5. Cost-sensitive? → DeepSeek-R1 or QwQ-32B

TOKEN BUDGET TIP:
  27-51% of reasoning tokens are FILLER (Hmm, Wait, Let me)
  Stripping filler doesn't affect accuracy!
  Monitor actual reasoning token consumption in production.
""")


> **What just happened?** Reasoning costs vary 100x between cheapest and most expensive options. The 5-step decision framework: try standard model first, add CoT if needed, escalate to reasoning models only for genuinely hard tasks. 27-51% of reasoning tokens are filler that does not affect accuracy. Always start cheap and escalate.


### Step 10 · Verification Chains — Generate, Check, Correct

Meta AI's CoVe: hallucinated entities dropped from 2.95 to 0.68.

**`verification.py`** · _Block 12 of 12_

CHAIN OF VERIFICATION (CoVe) — Meta AI 2023


In [ ]:
# =============================================
# CHAIN OF VERIFICATION (CoVe) — Meta AI 2023
# =============================================

# Step 1: Draft initial response
draft = ask("List 5 important sections of the Indian IT Act 2000")

# Step 2: Generate fact-checking questions
questions = ask(f"Generate 5 fact-check questions for: {draft}")

# Step 3: Answer INDEPENDENTLY (don't show original draft!)
# This prevents copying the same hallucination
verified = ask(f"Answer these questions independently: {questions}")

# Step 4: Produce verified response
final = ask(f"Revise based on verified facts: {verified}")

print("""
CoVe RESULTS:
  Hallucinated entities: 2.95 → 0.68 (77% reduction!)
  Biography accuracy: 55.9% → 71.4%
  
  Cost: 3-4 LLM calls per query
  Use for: Legal citations, medical info, financial data
  
  CRITICAL FOR INDIA:
  LLMs hallucinate Indian case citations frequently.
  Never trust model-generated section numbers (IPC/BNS).
  Always verify against primary legal databases.
""")


> **What just happened?** Chain of Verification (Meta AI): generate → fact-check → answer independently → revise. Hallucinations dropped 77%. The key: Step 3 answers without seeing the original draft to avoid copying errors. Critical for Indian applications: LLMs frequently hallucinate case citations and statute sections.


---

## Wrap-up

You've walked through all 12 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
